In [1]:
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages

In [2]:
load_dotenv()

True

In [ ]:
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing")

postgres_uri = os.getenv("POSTGRES_URI")
if not postgres_uri:
    raise ValueError("POSTGRES_URI is missing")

In [4]:
primary = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=google_api_key,
    max_retries=3,
)
backup = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    google_api_key=google_api_key,
    max_retries=3,
)

model = primary.with_fallbacks([backup])

In [5]:
class MemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
def chat_model(state: MemoryState):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

In [7]:
checkpointer = PostgresSaver.from_conn_string(postgres_uri)
checkpointer.setup()

graph = StateGraph(MemoryState)
graph.add_node("chat_model", chat_model)
graph.add_edge(START, "chat_model")

bot = graph.compile(checkpointer=checkpointer)

In [8]:
config = {"configurable": {"thread_id": "thread_postgres_1"}}
response = bot.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    config=config,
)
response


In [9]:
response = bot.invoke(
    {"messages": [HumanMessage(content="What did I ask you just now?")]},
    config=config,
)
response